# K3D-02 v3 · Цели для студента: teacher на train-чанке (две фазы)

**Запускать только коммитом: Save Version → Save & Run All (Commit).** Интерактивный Run All умирает вместе с сессией редактора, и результаты пропадают. Ячейка 0 сама остановит запуск в редакторе.

**Фаза 1 (~1.2 ч):** для каждого объекта чанка до 45k вокселей — одношаговые предсказания teacher (2 NFE). Файлы `targets/x0/<id>.npz`:
- `x0c` — предсказание при t=1 с картинкой; `x0u` — без картинки (только геометрия). Одношаговый teacher с CFG w = `x0c + w·(x0c − x0u)`;
- `z` — исходный шум (seed 42, генератор T4).

**Фаза 2 (~7 ч):** полный teacher за 8 шагов (CFG 7, seed 42) для объектов до 20k вокселей, в случайном порядке, пока не кончится бюджет `MAX_HOURS`. Файлы `targets/lat8/<id>.npz` (`latent`). Студенту не нужна: `PHASE2 = False` в ячейке 0 её выключает.

Все массивы [N,16] fp16, строки в порядке `vox_idxi`. Данные качаются потоком (только `cache/`), GPU начинают считать, не дожидаясь конца загрузки.

**Настройки Kaggle:** Accelerator **GPU T4 x2**, Internet **On**. После запуска коммита останови сессию редактора — она тратит квоту.

**После завершения (~8.5 ч):** открой сохранённую версию → Output → скачай `targets.zip` (~1.5 ГБ) в папку `3D DIT Distillation`. Output коммита хранится постоянно, спешить не нужно.

**Продолжение:** Add Input → Notebook Output → эта версия; в ячейке 0 укажи `PREV_TARGETS` (путь вида `/kaggle/input/<имя>/targets`).

In [ ]:
# 0. Конфигурация
import os, sys, subprocess, json, time, shutil, glob, csv
from pathlib import Path

CHUNK = 1            # номер train-чанка (1..15)
STEPS = 8            # шаги teacher для целей (8 -> LPIPS ~0.016 к 50 шагам)
MAX_HOURS = 9.0      # бюджет на GPU-воркер (коммит Kaggle <= 12 ч; фаза 1 ~1.2 ч + фаза 2 ~7.2 ч)
MAX_VOX1 = 45000     # фаза 1 (1 шаг): объекты до этого размера (весь чанк)
MAX_VOX = 20000      # фаза 2 (8 шагов): объекты до этого размера
PHASE2 = True        # False: только фаза 1 (x0) — её достаточно для поставленного студента (~1.2 ч)
PREV_TARGETS = None  # например "/kaggle/input/k3d-02-teacher-targets/targets" для продолжения
ALLOW_INTERACTIVE = False  # True — только для отладки в редакторе

RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
print("режим запуска:", RUN_TYPE or "не Kaggle")
if RUN_TYPE == "Interactive" and not ALLOW_INTERACTIVE:
    raise RuntimeError("Это интерактивная сессия: результаты пропадут вместе с ней. "
                       "Запусти через Save Version → Save & Run All (Commit).")

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
K3D, DATA = WORK / "k3d", Path("/tmp/k3d")
REPO, WEIGHTS, STREAM = DATA / "3dgen-main", DATA / "weights", DATA / f"train_{CHUNK:03d}"
TARGETS = WORK / "targets"
for p in (K3D, DATA, STREAM, TARGETS):
    p.mkdir(parents=True, exist_ok=True)
if shutil.which("nvcc") is None and Path("/usr/local/cuda/bin/nvcc").exists():
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
    os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpus", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Нет GPU: включи Accelerator = GPU T4 x2"


In [ ]:
# 1. Установка зависимостей (1–3 мин); сетевые сбои повторяются
import importlib

def pip(*args, tries=3):
    for k in range(tries):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
        if r.returncode == 0:
            print("OK   " + " ".join(args)[:150]); return True
        net = any(x in r.stderr for x in ("name resolution", "Failed to establish", "Read timed out", "ConnectionError"))
        if net and k + 1 < tries:
            print(f"     сеть недоступна, повтор через 15 с ({k + 1}/{tries})"); time.sleep(15); continue
        print("FAIL " + " ".join(args)[:150]); print(r.stderr[-800:]); return False

for pkg in ["ninja", "einops", "trimesh", "safetensors", "scikit-learn", "pandas"]:
    pip(pkg)
pip("diffusers==0.35.1") or pip("diffusers")
try:
    from transformers import PretrainedConfig  # noqa: F401
except Exception as exc:
    print("transformers не подходит:", exc); pip("transformers==4.57.3")
cu = (torch.version.cuda or "12.4").replace(".", "")
# spconv не ставим: на этом пути он не нужен (t4compat подставит заглушку), а его cumm пытается собраться при импорте
tv = torch.__version__.split("+")[0]
pip("torch-scatter", "--only-binary=:all:", "-f", f"https://data.pyg.org/whl/torch-{'.'.join(tv.split('.')[:2])}.0+cu{cu}.html") \
    or print("-> torch_scatter: будет pure-torch shim")
importlib.invalidate_caches()
missing = [m for m in ["ninja", "einops", "trimesh", "safetensors", "sklearn", "pandas", "diffusers", "transformers"]
           if importlib.util.find_spec(m) is None]
print("не хватает модулей:", missing or "нет")
assert not missing, f"Перезапусти ячейку: не установились {missing}"


## 2. Исходники
Ячейки ниже только записывают файлы в `/kaggle/working/k3d/`.

In [ ]:
# 2.0 папки
for d in ['calib', 'gen', 't4compat', 't4compat/shim_flash/flash_attn', 't4compat/shim_flash/flash_attn/ops', 't4compat/shim_flash/flash_attn/ops/triton', 't4compat/shim_nvdiffrast/nvdiffrast', 't4compat/shim_nvdiffrast/nvdiffrast/torch', 't4compat/shims_scatter/torch_scatter', 't4compat/shims_spconv/spconv', 't4compat/shims_spconv/spconv/pytorch']:
    (K3D / d).mkdir(parents=True, exist_ok=True)
print('ok')

In [ ]:
%%writefile /kaggle/working/k3d/t4compat/__init__.py
"""Run the Kandinsky3D lat_dit baseline on GPUs it was not built for (Kaggle T4, sm75) and on CPU.

Usage (before importing anything from the baseline repo):

    import t4compat
    report = t4compat.install(repo_root="/path/to/3dgen-main")   # shims + dtype policy
    report.update(t4compat.prepare_ops())                        # JIT-compile CUDA ops or fall back
    model = t4compat.load_lat_dit_fp16(".../lat_dit", device)    # 1.95B teacher in fp16, low RAM peak

What it changes:
  * flash_attn      -> SDPA-based shim (FlashAttention-2 does not run on sm75)
  * nvdiffrast      -> stub if not installed (only imported, never used on this path)
  * torch_scatter   -> pure-torch shim if no wheel matches the installed torch
  * spconv          -> data-holder stub if not installed (no spconv convolutions on this path)
  * lat_dit autocast dtype bf16 -> fp16 (T4 has no fast bf16); residual stream kept in fp32
  * custom CUDA ops (curope, libsparse conv, triton ordering, FlexGEMM) -> pure-torch fallbacks when unavailable
"""

from __future__ import annotations

import os
import sys
import traceback
from pathlib import Path
from typing import Dict, Optional

import torch

HERE = Path(__file__).resolve().parent
__all__ = ["install", "prepare_ops", "load_lat_dit_fp16", "wrap_fp32_residual", "gpu_capability"]


def _prepend(path: Path) -> None:
    p = str(path)
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)


def gpu_capability():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_capability()


def _has_working_flash_attn() -> bool:
    cap = gpu_capability()
    if cap is None or cap[0] < 8:
        return False
    try:
        import flash_attn  # noqa: F401
        return not getattr(flash_attn, "__version__", "").endswith("sdpa-shim")
    except Exception:
        return False


def install(repo_root: str, attn_shim: Optional[bool] = None, fp16_lat_dit: Optional[bool] = None,
            allow_spconv_stub: Optional[bool] = None, verbose: bool = True) -> Dict[str, str]:
    report: Dict[str, str] = {}
    cpu = gpu_capability() is None
    if attn_shim is None:
        attn_shim = not _has_working_flash_attn()
    if fp16_lat_dit is None:
        cap = gpu_capability()
        fp16_lat_dit = cap is None or cap[0] < 8
    if allow_spconv_stub is None:
        # on this code path spconv is only a data holder for SparseTensor (no spconv convolutions are called)
        allow_spconv_stub = True

    if attn_shim:
        for name in [m for m in sys.modules if m == "flash_attn" or m.startswith("flash_attn.")]:
            del sys.modules[name]
        _prepend(HERE / "shim_flash")
        report["flash_attn"] = "sdpa-shim"
    else:
        report["flash_attn"] = "native"

    try:
        import nvdiffrast.torch  # noqa: F401
        report["nvdiffrast"] = "native"
    except Exception:
        _prepend(HERE / "shim_nvdiffrast")
        report["nvdiffrast"] = "stub"

    try:
        import torch_scatter  # noqa: F401
        report["torch_scatter"] = "native"
    except Exception:
        _prepend(HERE / "shims_scatter")
        report["torch_scatter"] = "torch-shim"

    try:
        import spconv.pytorch  # noqa: F401
        report["spconv"] = "native"
    except Exception:
        if not allow_spconv_stub:
            raise
        _prepend(HERE / "shims_spconv")
        report["spconv"] = "stub"
        if not cpu:
            print("[t4compat] WARNING: spconv not installed, using the data-holder stub")

    _prepend(Path(repo_root))
    import kandi3d_image23d.modeling_utils as mu
    if fp16_lat_dit:
        mu.STAGE_COMPUTE_DTYPES["lat_dit"] = torch.float16
    report["lat_dit_autocast"] = str(mu.STAGE_COMPUTE_DTYPES["lat_dit"]).replace("torch.", "")
    if verbose:
        print("[t4compat] install:", report)
    return report


# ----------------------------------------------------------------------------- op fallbacks
def patch_rope() -> None:
    import importlib
    from .fallback_ops import rope_nd_torch

    def rope_nd(x, pos, freq=100.0):
        return rope_nd_torch(x, pos, freq)

    import ops.rope as R
    R.rope_nd = rope_nd
    for name in ("models.generators.lat_dit", "models.encoders_2D.fc_vae"):
        m = importlib.import_module(name)
        m.rope_nd = rope_nd


def patch_sparse_conv() -> None:
    from .fallback_ops import conv_torch
    import ops.sparse as S
    S.conv = conv_torch


def patch_order() -> None:
    from .fallback_ops import encode_hilbert_torch, encode_z_torch
    import base_models.order as O
    O.encode_hilbert = encode_hilbert_torch
    O.encode_z = encode_z_torch


def flexconv_torch_forward(self, feats, coords, shape, neighbours_cache=None):
    """Gather-matmul replacement for base_models.sparse.FlexConv3d (weight [Co, 3, 3, 3, Ci]).
    Offset convention assumed: weight[:, i, j, k, :] <-> (dx, dy, dz) = (i-1, j-1, k-1) on coords[:, 1:4]."""
    from .fallback_ops import conv_neighbors_torch
    if neighbours_cache is None:
        neighbours_cache = conv_neighbors_torch(coords)
    w = self.weight.float()
    Co, Ci = w.shape[0], w.shape[-1]
    w = w.reshape(Co, 27, Ci)
    out = self.bias.float().expand(len(feats), Co).clone()
    f = feats.float()
    for d in range(27):
        idx = neighbours_cache[d]
        m = idx >= 0
        if bool(m.any()):
            out[m] += f[idx[m]] @ w[:, d, :].t()
    return out.to(feats.dtype), neighbours_cache


def patch_flexconv() -> None:
    import base_models.sparse as sp
    sp.FlexConv3d.forward = flexconv_torch_forward


def prepare_ops(force_fallback: bool = False, verbose: bool = True, need_mesh: bool = True) -> Dict[str, str]:
    """JIT-compile the repo CUDA extensions; fall back to pure torch per op when that fails.
    need_mesh=False skips libmesh (only needed to decode/render meshes), saving ~5-10 min of compilation."""
    report: Dict[str, str] = {}
    cpu = gpu_capability() is None
    if cpu or force_fallback:
        patch_rope(); patch_sparse_conv(); patch_order(); patch_flexconv()
        report.update(rope="torch", sparse_conv="torch", order="torch", flexconv="torch", libmesh="unavailable")
        if verbose:
            print("[t4compat] ops:", report)
        return report

    os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "{}.{}".format(*gpu_capability()))
    import torch.utils.cpp_extension as cpp_ext
    if not hasattr(cpp_ext, "_get_build_directory"):  # private helper used by ops/custom_ops.py
        def _get_build_directory(name, verbose=False):
            root = os.environ.get("TORCH_EXTENSIONS_DIR", os.path.expanduser("~/.cache/torch_extensions"))
            path = os.path.join(root, name)
            os.makedirs(path, exist_ok=True)
            return path
        cpp_ext._get_build_directory = _get_build_directory
    targets = [("rope", "ops.rope", patch_rope), ("sparse_conv", "ops.sparse", patch_sparse_conv)]
    if need_mesh:
        targets.append(("libmesh", "ops.mesh", None))
    else:
        report["libmesh"] = "skipped"
    for name, mod, patch in targets:
        try:
            import importlib
            importlib.import_module(mod)._init()
            report[name] = "cuda"
        except Exception as exc:  # noqa: BLE001
            print(f"[t4compat] compiling {mod} failed: {type(exc).__name__}: {exc}")
            traceback.print_exc(limit=2)
            if patch is not None:
                patch()
                report[name] = "torch"
            else:
                report[name] = "unavailable"
    try:  # triton ordering kernels
        import base_models.order as O
        xyz = torch.zeros(8, 4, dtype=torch.int32, device="cuda")
        O.encode_hilbert(xyz)
        report["order"] = "triton"
    except Exception as exc:  # noqa: BLE001
        print(f"[t4compat] triton ordering failed ({exc}); using torch")
        patch_order()
        report["order"] = "torch"
    try:  # FlexGEMM (used by the lat_vae decoder)
        import flex_gemm  # noqa: F401
        report["flexconv"] = "flex_gemm"
    except Exception:
        patch_flexconv()
        report["flexconv"] = "torch"
    if verbose:
        print("[t4compat] ops:", report)
    return report


# ----------------------------------------------------------------------------- model loading
class _Fp32Out(torch.nn.Module):
    """Wraps flow.patchify so the transformer residual streams (x and cond) stay in fp32 under fp16 autocast."""

    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x):
        out = self.inner(x)
        return out.replace(out.feats.float())


def wrap_fp32_residual(lat_dit_model) -> None:
    flow = lat_dit_model.core.flow
    if not isinstance(flow.patchify, _Fp32Out):
        flow.patchify = _Fp32Out(flow.patchify)


def load_lat_dit_fp16(weights_dir: str, device, fp32_residual: bool = True, keep_fp32=("mean", "std"),
                      dtype: torch.dtype = torch.float16):
    """Build LatDitModel on the meta device and fill it tensor-by-tensor in `dtype` (fp16: RAM peak ~4 GB, not ~16 GB).
    Use dtype=torch.float32 on CPU (no autocast there)."""
    from safetensors import safe_open
    from kandi3d_image23d.models.modeling_lat_dit import (LatDitModel, _config_to_init_kwargs,
                                                          _load_lat_dit_config)
    path = Path(weights_dir)
    cfg = _load_lat_dit_config(path)
    with torch.device("meta"):
        model = LatDitModel(**_config_to_init_kwargs(cfg))
    sd = {}
    with safe_open(str(path / "diffusion_pytorch_model.safetensors"), framework="pt", device="cpu") as f:
        for k in f.keys():
            t = f.get_tensor(k)
            if t.is_floating_point() and k.replace("_orig_mod.", "") not in keep_fp32:
                t = t.to(dtype)
            sd[k] = t
    missing, unexpected = model.load_state_dict(sd, strict=False, assign=True)
    meta_left = [n for n, p in list(model.named_parameters()) + list(model.named_buffers()) if p.is_meta]
    if meta_left:
        raise RuntimeError(f"parameters not loaded: {meta_left[:10]} (missing={missing[:10]})")
    if unexpected:
        print("[t4compat] unexpected keys:", unexpected[:10])
    model.to(torch.device(device))
    model.eval()
    if fp32_residual:
        wrap_fp32_residual(model)
    return model


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/fallback_ops.py
"""Pure-PyTorch versions of the baseline's custom CUDA / Triton ops.

They reproduce the semantics of
  ops/curope/rope.cu            -> rope_nd_torch
  ops/libsparse/conv.cu (fwd)   -> sparse_dwconv3d_torch  (depthwise 3x3x3 submanifold conv)
  base_models/order.py (triton) -> encode_hilbert_torch / encode_z_torch
and are used (a) for CPU smoke tests and (b) as a fallback if JIT compilation fails on the GPU.
"""

from __future__ import annotations

from typing import Optional

import torch


# ----------------------------------------------------------------------------- RoPE
def rope_nd_torch(x: torch.Tensor, pos: torch.Tensor, freq: float = 100.0, F0: float = 1.0) -> torch.Tensor:
    """x: [B, N, H, D], pos: [B, N, DIM] -> rotated x (same dtype).

    Head dim is split into DIM chunks of size 2Q (Q = D / (2*DIM)); chunk X = [u (Q) | v (Q)],
    angle_j = pos[X] * F0 * freq^(-j/Q), u' = u cos - v sin, v' = v cos + u sin.
    """
    B, N, H, D = x.shape
    DIM = pos.shape[-1]
    Q = D // (2 * DIM)
    j = torch.arange(Q, device=x.device, dtype=torch.float64)
    inv = F0 * torch.pow(torch.tensor(float(freq), dtype=torch.float64, device=x.device), -j / Q)
    ang = pos.to(torch.float64)[..., None] * inv  # [B, N, DIM, Q]
    cos = ang.cos().to(torch.float32)[:, :, None]  # [B, N, 1, DIM, Q]
    sin = ang.sin().to(torch.float32)[:, :, None]
    xv = x.to(torch.float32).reshape(B, N, H, DIM, 2, Q)
    u, v = xv[..., 0, :], xv[..., 1, :]
    out = torch.stack([u * cos - v * sin, v * cos + u * sin], dim=-2)
    return out.reshape(B, N, H, D).to(x.dtype)


# ----------------------------------------------------------------------------- sparse depthwise conv
def _keys(coords: torch.Tensor) -> torch.Tensor:
    c = coords.long()
    return (c[:, 0] << 36) | (c[:, 1] << 24) | (c[:, 2] << 12) | c[:, 3]


def conv_neighbors_torch(coords: torch.Tensor) -> torch.Tensor:
    """coords [N, 4] (b, x, y, z) -> neighbour index [27, N] (-1 if absent).

    Offset order follows conv.cu: d -> (dx, dy, dz) = (d // 9 - 1, (d // 3) % 3 - 1, d % 3 - 1).
    """
    keys = _keys(coords)
    sorted_keys, order = torch.sort(keys)
    c = coords.long()
    out = torch.full((27, len(coords)), -1, dtype=torch.long, device=coords.device)
    for d in range(27):
        dx, dy, dz = d // 9 - 1, (d // 3) % 3 - 1, d % 3 - 1
        x, y, z = c[:, 1] + dx, c[:, 2] + dy, c[:, 3] + dz
        valid = (x >= 0) & (y >= 0) & (z >= 0)
        nk = (c[:, 0] << 36) | (x.clamp(min=0) << 24) | (y.clamp(min=0) << 12) | z.clamp(min=0)
        p = torch.searchsorted(sorted_keys, nk).clamp(max=len(keys) - 1)
        found = valid & (sorted_keys[p] == nk)
        out[d] = torch.where(found, order[p], torch.full_like(p, -1))
    return out


def sparse_dwconv3d_torch(feat: torch.Tensor, coords: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor,
                          neighbors: Optional[torch.Tensor] = None, chunk: int = 4096) -> torch.Tensor:
    """feat [N, C], coords [N, 4], weight [3, 3, 3, C], bias [C] -> [N, C] (feat dtype, fp32 accumulation)."""
    if neighbors is None:
        neighbors = conv_neighbors_torch(coords)
    w = weight.reshape(27, -1).float()
    b = bias.float()
    N, C = feat.shape
    out = torch.empty_like(feat)
    for s in range(0, N, chunk):
        e = min(N, s + chunk)
        acc = b.expand(e - s, C).clone()
        for d in range(27):
            idx = neighbors[d, s:e]
            m = idx >= 0
            if not bool(m.any()):
                continue
            g = feat[idx.clamp(min=0)].float() * m[:, None]
            acc.add_(g * w[d])
        out[s:e] = acc.to(feat.dtype)
    return out


def conv_torch(feat, pos, weight, bias, indices=None):
    """Drop-in replacement for ops.sparse.conv (forward only)."""
    return sparse_dwconv3d_torch(feat, pos, weight, bias)


# ----------------------------------------------------------------------------- serialization codes
def _cols(xyz: torch.Tensor, convention: str):
    xo, yo, zo = convention.find("x"), convention.find("y"), convention.find("z")
    c = xyz.long()
    return c[:, 0], c[:, 1 + xo], c[:, 1 + yo], c[:, 1 + zo]


def encode_hilbert_torch(xyz: torch.Tensor, convention: str = "xyz", assign_batch_index: bool = True) -> torch.Tensor:
    b, x, y, z = _cols(xyz, convention)
    x, y, z = x.clone(), y.clone(), z.clone()
    zero = torch.zeros_like(x)
    for i in range(15, 0, -1):
        q = 1 << i
        p = q - 1
        x = x ^ torch.where((x & q) > 0, zero + p, zero)
        cond = (y & q) > 0
        t = (x ^ y) & p
        x = x ^ torch.where(cond, zero + p, t)
        y = y ^ torch.where(cond, zero, t)
        cond = (z & q) > 0
        t = (x ^ z) & p
        x = x ^ torch.where(cond, zero + p, t)
        z = z ^ torch.where(cond, zero, t)
    y = y ^ x
    z = z ^ y
    t = zero.clone()
    for i in range(15, 0, -1):
        q = 1 << i
        t = t ^ torch.where((z & q) > 0, zero + (q - 1), zero)
    x, y, z = x ^ t, y ^ t, z ^ t
    ret = zero.clone()
    for i in range(16):
        q = 1 << i
        ret = ret | ((x & q) << (2 * i + 2)) | ((y & q) << (2 * i + 1)) | ((z & q) << (2 * i))
    if assign_batch_index:
        ret = ret | (b << 48)
    return ret


def encode_z_torch(xyz: torch.Tensor, convention: str = "xyz", assign_batch_index: bool = True) -> torch.Tensor:
    b, x, y, z = _cols(xyz, convention)
    ret = torch.zeros_like(x)
    for i in range(16):
        q = 1 << i
        ret = ret | ((x & q) << (2 * i + 2)) | ((y & q) << (2 * i + 1)) | ((z & q) << (2 * i))
    if assign_batch_index:
        ret = ret | (b << 48)
    return ret


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/__init__.py
"""Drop-in replacement for the parts of `flash_attn` used by the Kandinsky3D baseline.

Used on GPUs without FlashAttention-2 support (e.g. Kaggle T4, sm75) and on CPU.
Everything is implemented with torch.nn.functional.scaled_dot_product_attention,
which picks the memory-efficient kernel on T4 and the flash kernel on A100.

Tensor layouts follow flash_attn:
  dense:  q/k/v [B, S, H, D], qkv [B, S, 3, H, D]
  varlen: q/k/v [T, H, D], qkv [T, 3, H, D], kv [T, 2, H, D], cu_seqlens int32 [n+1]
"""

from __future__ import annotations

from typing import List, Optional, Tuple

import torch
import torch.nn.functional as F

__version__ = "0.0.0+sdpa-shim"


def _check_unsupported(softcap: float = 0.0, window_size=(-1, -1), dropout_p: float = 0.0) -> None:
    if softcap not in (0, 0.0, None):
        raise NotImplementedError("flash_attn shim: softcap is not supported")
    if window_size is not None and tuple(window_size) not in ((-1, -1),):
        raise NotImplementedError(f"flash_attn shim: window_size={window_size} is not supported")
    if dropout_p not in (0, 0.0, None):
        raise NotImplementedError("flash_attn shim: dropout is not supported")


def _sdpa(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, scale: Optional[float], causal: bool) -> torch.Tensor:
    """q/k/v in [B, S, H, D] -> out [B, S, H, D]."""
    out = F.scaled_dot_product_attention(
        q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
        is_causal=causal, scale=scale,
    )
    return out.transpose(1, 2)


def flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=None, causal=False,
                    window_size=(-1, -1), softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    return _sdpa(q, k, v, softmax_scale, causal)


def flash_attn_qkvpacked_func(qkv, dropout_p=0.0, softmax_scale=None, causal=False,
                              window_size=(-1, -1), softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    q, k, v = qkv.unbind(dim=2)
    return _sdpa(q, k, v, softmax_scale, causal)


def flash_attn_kvpacked_func(q, kv, dropout_p=0.0, softmax_scale=None, causal=False,
                             window_size=(-1, -1), softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    k, v = kv.unbind(dim=2)
    return _sdpa(q, k, v, softmax_scale, causal)


def _lengths(cu_seqlens: torch.Tensor) -> List[int]:
    cu = cu_seqlens.detach().to("cpu", torch.int64).tolist()
    return [cu[i + 1] - cu[i] for i in range(len(cu) - 1)]


def _runs(lq: List[int], lk: List[int]) -> List[Tuple[int, int, int]]:
    """Group consecutive segments with identical (len_q, len_k): returns (start_seg, n_seg, ...)."""
    runs = []
    i = 0
    while i < len(lq):
        j = i + 1
        while j < len(lq) and lq[j] == lq[i] and lk[j] == lk[i]:
            j += 1
        runs.append((i, j - i))
        i = j
    return runs


def _varlen(q, k, v, cu_q, cu_k, scale, causal):
    lq, lk = _lengths(cu_q), _lengths(cu_k)
    assert len(lq) == len(lk), "cu_seqlens_q and cu_seqlens_k describe different batch sizes"
    out = torch.empty_like(q)
    oq, ok = 0, 0
    for start, n in _runs(lq, lk):
        Lq, Lk = lq[start], lk[start]
        if Lq == 0:
            ok += n * Lk
            continue
        qs = q[oq:oq + n * Lq].reshape(n, Lq, *q.shape[1:])
        ks = k[ok:ok + n * Lk].reshape(n, Lk, *k.shape[1:])
        vs = v[ok:ok + n * Lk].reshape(n, Lk, *v.shape[1:])
        out[oq:oq + n * Lq] = _sdpa(qs, ks, vs, scale, causal).reshape(n * Lq, *q.shape[1:])
        oq += n * Lq
        ok += n * Lk
    return out


def flash_attn_varlen_func(q, k, v, cu_seqlens_q, cu_seqlens_k, max_seqlen_q=None, max_seqlen_k=None,
                           dropout_p=0.0, softmax_scale=None, causal=False, window_size=(-1, -1),
                           softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    return _varlen(q, k, v, cu_seqlens_q, cu_seqlens_k, softmax_scale, causal)


def flash_attn_varlen_qkvpacked_func(qkv, cu_seqlens, max_seqlen=None, dropout_p=0.0, softmax_scale=None,
                                     causal=False, window_size=(-1, -1), softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    q, k, v = qkv.unbind(dim=1)
    return _varlen(q, k, v, cu_seqlens, cu_seqlens, softmax_scale, causal)


def flash_attn_varlen_kvpacked_func(q, kv, cu_seqlens_q, cu_seqlens_k, max_seqlen_q=None, max_seqlen_k=None,
                                    dropout_p=0.0, softmax_scale=None, causal=False, window_size=(-1, -1),
                                    softcap=0.0, **kwargs):
    _check_unsupported(softcap, window_size, dropout_p)
    k, v = kv.unbind(dim=1)
    return _varlen(q, k, v, cu_seqlens_q, cu_seqlens_k, softmax_scale, causal)


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/__init__.py
"""flash_attn.ops shim package."""


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/_norms.py
"""Reference LayerNorm / RMSNorm with fp32 accumulation, returning the input dtype."""
import torch
import torch.nn.functional as F


def layer_norm_ref(x, weight=None, bias=None, eps=1e-6):
    dtype = x.dtype
    w = None if weight is None else weight.float()
    b = None if bias is None else bias.float()
    return F.layer_norm(x.float(), (x.shape[-1],), w, b, eps).to(dtype)


def rms_norm_ref(x, weight=None, bias=None, eps=1e-6):
    dtype = x.dtype
    xf = x.float()
    y = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + eps)
    if weight is not None:
        y = y * weight.float()
    if bias is not None:
        y = y + bias.float()
    return y.to(dtype)


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/activations.py
"""swiglu as in flash_attn.ops.activations: silu(x) * y."""
import torch.nn.functional as F


def swiglu(x, y):
    return F.silu(x) * y


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/layer_norm.py
"""flash_attn.ops.layer_norm.layer_norm(x, weight, bias, epsilon)."""
from flash_attn.ops._norms import layer_norm_ref


def layer_norm(x, weight, bias, epsilon=1e-6, *args, **kwargs):
    return layer_norm_ref(x, weight, bias, epsilon)


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/rms_norm.py
"""flash_attn.ops.rms_norm.rms_norm(x, weight, epsilon)."""
from flash_attn.ops._norms import rms_norm_ref


def rms_norm(x, weight, epsilon=1e-6, *args, **kwargs):
    return rms_norm_ref(x, weight, None, epsilon)


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/triton/__init__.py
"""flash_attn.ops.triton shim package."""


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_flash/flash_attn/ops/triton/layer_norm.py
"""flash_attn.ops.triton.layer_norm.{layer_norm_fn, rms_norm_fn} (no residual / dropout support)."""
from flash_attn.ops._norms import layer_norm_ref, rms_norm_ref


def _check(residual, dropout_p, prenorm, x1):
    if residual is not None or x1 is not None or (dropout_p or 0.0) != 0.0 or prenorm:
        raise NotImplementedError("flash_attn shim: residual/dropout/prenorm are not supported")


def layer_norm_fn(x, weight, bias, residual=None, x1=None, weight1=None, bias1=None, eps=1e-6,
                  dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                  is_rms_norm=False, return_dropout_mask=False, **kwargs):
    _check(residual, dropout_p, prenorm, x1)
    if is_rms_norm:
        return rms_norm_ref(x, weight, bias, eps)
    return layer_norm_ref(x, weight, bias, eps)


def rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None, eps=1e-6,
                dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                return_dropout_mask=False, **kwargs):
    _check(residual, dropout_p, prenorm, x1)
    return rms_norm_ref(x, weight, bias, eps)


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_nvdiffrast/nvdiffrast/__init__.py
"""Stub: the baseline imports nvdiffrast at module level but the lat_dit/lat_vae decode path
used here (renderer='smooth', extractor='fmc') never calls it."""


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shim_nvdiffrast/nvdiffrast/torch/__init__.py
class _Unavailable:
    def __init__(self, *args, **kwargs):
        raise RuntimeError("nvdiffrast stub: real nvdiffrast is not installed")


RasterizeCudaContext = _Unavailable
RasterizeGLContext = _Unavailable


def _unavailable(*args, **kwargs):
    raise RuntimeError("nvdiffrast stub: real nvdiffrast is not installed")


rasterize = interpolate = texture = antialias = _unavailable


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shims_scatter/torch_scatter/__init__.py
"""Minimal pure-PyTorch replacement for torch_scatter (scatter_add / scatter_sum / scatter_mean / scatter_max).

Only used when no prebuilt torch_scatter wheel matches the installed torch.
Semantics follow torch_scatter: `index` is broadcast to `src` along `dim`, `out` is updated in place.
"""

from typing import Optional, Tuple

import torch

__version__ = "0.0.0+torch-shim"


def _broadcast(index: torch.Tensor, src: torch.Tensor, dim: int) -> torch.Tensor:
    if dim < 0:
        dim = src.dim() + dim
    if index.dim() == 1:
        shape = [1] * src.dim()
        shape[dim] = -1
        index = index.view(shape)
    for _ in range(index.dim(), src.dim()):
        index = index.unsqueeze(-1)
    return index.expand_as(src)


def _make_out(src, index, dim, out, dim_size, fill=0):
    if out is not None:
        return out
    if dim < 0:
        dim = src.dim() + dim
    size = list(src.shape)
    if dim_size is not None:
        size[dim] = dim_size
    elif index.numel() == 0:
        size[dim] = 0
    else:
        size[dim] = int(index.max()) + 1
    return torch.full(size, fill, dtype=src.dtype, device=src.device)


def scatter_sum(src: torch.Tensor, index: torch.Tensor, dim: int = -1, out: Optional[torch.Tensor] = None,
                dim_size: Optional[int] = None) -> torch.Tensor:
    index_b = _broadcast(index.long(), src, dim)
    out = _make_out(src, index_b, dim, out, dim_size)
    return out.scatter_add_(dim, index_b, src.to(out.dtype))


scatter_add = scatter_sum


def scatter_mean(src: torch.Tensor, index: torch.Tensor, dim: int = -1, out: Optional[torch.Tensor] = None,
                 dim_size: Optional[int] = None) -> torch.Tensor:
    out = scatter_sum(src, index, dim, out, dim_size)
    d = dim if dim >= 0 else src.dim() + dim
    count = scatter_sum(torch.ones_like(src, dtype=out.dtype), index, d, None, out.shape[d])
    count = count.clamp(min=1)
    if out.is_floating_point():
        out.div_(count)
    else:
        out.floor_divide_(count)
    return out


def scatter_max(src: torch.Tensor, index: torch.Tensor, dim: int = -1, out: Optional[torch.Tensor] = None,
                dim_size: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    index_b = _broadcast(index.long(), src, dim)
    if out is None:
        out = _make_out(src, index_b, dim, None, dim_size, fill=float("-inf") if src.is_floating_point() else torch.iinfo(src.dtype).min)
        out.scatter_reduce_(dim, index_b, src, reduce="amax", include_self=False)
        if out.is_floating_point():
            out.masked_fill_(torch.isinf(out) & (out < 0), 0)
    else:
        out.scatter_reduce_(dim, index_b, src.to(out.dtype), reduce="amax", include_self=True)
    argmax = torch.zeros_like(out, dtype=torch.long)
    return out, argmax


def scatter_min(src: torch.Tensor, index: torch.Tensor, dim: int = -1, out: Optional[torch.Tensor] = None,
                dim_size: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    out_neg, arg = scatter_max(-src, index, dim, None if out is None else -out, dim_size)
    res = -out_neg
    if out is not None:
        out.copy_(res)
        res = out
    return res, arg


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shims_spconv/spconv/__init__.py
"""CPU-test stub package for spconv."""


In [ ]:
%%writefile /kaggle/working/k3d/t4compat/shims_spconv/spconv/pytorch/__init__.py
"""CPU-test stub of spconv.pytorch.SparseConvTensor (data holder only, no convolutions)."""

import torch


class SparseConvTensor:
    def __init__(self, features, indices, spatial_shape, batch_size, grid=None, voxel_num=None,
                 indice_dict=None, benchmark=False, permanent_thrust_allocator=False,
                 enable_timer=False, force_algo=None, *args, **kwargs):
        self._features = features
        self.indices = indices
        self.spatial_shape = [int(s) for s in spatial_shape]
        self.batch_size = batch_size
        self.grid = grid
        self.voxel_num = voxel_num
        self.indice_dict = {} if indice_dict is None else indice_dict
        self.benchmark = benchmark
        self.benchmark_record = {}
        self.thrust_allocator = None
        self._timer = None
        self.force_algo = force_algo
        self.int8_scale = None

    @property
    def features(self):
        return self._features

    @features.setter
    def features(self, value):
        self._features = value

    def dense(self, channels_first=True):
        raise NotImplementedError("spconv stub: dense() not available")


In [ ]:
%%writefile /kaggle/working/k3d/calib/k3d_data.py
"""Download helpers for the competition data hosted on sbercloud OBS (S3-compatible, supports HTTP Range).

* extract_from_remote_zip(url, dest, member_filter): pulls only the selected members of a remote zip
  via Range requests (falls back to a full download when the server ignores Range).
* stream_tar_members(url, dest, member_filter, stop_after): streams a remote tar and keeps matching
  members; can stop early, so a 16 GB chunk does not have to be stored.
"""

from __future__ import annotations

import io
import os
import shutil
import struct
import tarfile
import time
import zipfile
from pathlib import Path
from typing import Callable, Iterable, List, Optional

import requests

BASE = "https://rndml-team-xr.obs.ru-moscow-1.hc.sbercloud.ru/mazurov/tmp_data/AIJC_data"
URLS = {
    "code": f"{BASE}/3dgen-main.zip",
    "weights": f"{BASE}/kandi3d_weights_aijc_kit.zip",
    "public_test": f"{BASE}/objhq_splits_clean/public_test.zip",
    "train": f"{BASE}/objhq_splits_clean/train_archives/train_part_{{:03d}}.tar",
}


def _session() -> requests.Session:
    s = requests.Session()
    s.headers["User-Agent"] = "k3d-kaggle/1.0"
    return s


class HttpRangeReader(io.RawIOBase):
    """Seekable read-only file over HTTP Range requests with a simple read-ahead buffer."""

    def __init__(self, url: str, chunk: int = 16 << 20, session: Optional[requests.Session] = None, retries: int = 5):
        self.url, self.chunk, self.retries = url, chunk, retries
        self.s = session or _session()
        r = self.s.head(url, allow_redirects=True, timeout=60)
        r.raise_for_status()
        self.size = int(r.headers["Content-Length"])
        self.pos = 0
        self._buf, self._buf_start = b"", -1

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def fetch(self, start: int, end_incl: int) -> bytes:
        for attempt in range(self.retries):
            try:
                r = self.s.get(self.url, headers={"Range": f"bytes={start}-{end_incl}"}, timeout=300)
                if r.status_code != 206:
                    raise IOError(f"server ignored Range (status {r.status_code})")
                data = r.content
                if len(data) != end_incl - start + 1:
                    raise IOError("short read")
                return data
            except (requests.RequestException, IOError):
                if attempt == self.retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        raise IOError("unreachable")

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = max(0, min(n, self.size - self.pos))
        if n == 0:
            return b""
        bs = self._buf_start
        if bs >= 0 and bs <= self.pos and self.pos + n <= bs + len(self._buf):
            o = self.pos - bs
            data = self._buf[o:o + n]
        elif n >= self.chunk:
            data = self.fetch(self.pos, self.pos + n - 1)
        else:
            end = min(self.size, self.pos + self.chunk) - 1
            self._buf, self._buf_start = self.fetch(self.pos, end), self.pos
            data = self._buf[:n]
        self.pos += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[:len(data)] = data
        return len(data)


def _stream_range_to_file(sess, url, start, length, dst: Path, chunk=8 << 20):
    end = start + length - 1
    with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=600) as r:
        if r.status_code != 206:
            raise IOError(f"server ignored Range (status {r.status_code})")
        with open(dst, "wb") as f:
            for part in r.iter_content(chunk_size=chunk):
                f.write(part)
    if dst.stat().st_size != length:
        raise IOError(f"size mismatch for {dst}")


def _stream_deflate_to_file(sess, url, start, csize, usize, dst: Path, chunk=8 << 20):
    import zlib
    d = zlib.decompressobj(-15)
    end = start + csize - 1
    with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=600) as r:
        if r.status_code != 206:
            raise IOError(f"server ignored Range (status {r.status_code})")
        with open(dst, "wb") as f:
            for part in r.iter_content(chunk_size=chunk):
                f.write(d.decompress(part))
            f.write(d.flush())
    if dst.stat().st_size != usize:
        raise IOError(f"size mismatch for {dst}")


def download(url: str, dst: Path) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".part")
    with _session().get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 << 20)
    tmp.rename(dst)
    return dst


def list_remote_zip(url: str) -> List[zipfile.ZipInfo]:
    with zipfile.ZipFile(HttpRangeReader(url, chunk=1 << 20)) as zf:
        return zf.infolist()


def _target_name(name: str, strip_prefix: Optional[str], rename: Optional[Callable[[str], str]]) -> str:
    if rename is not None:
        return rename(name)
    if strip_prefix and name.startswith(strip_prefix):
        return name[len(strip_prefix):]
    return name


def extract_from_remote_zip(url: str, dest: Path, member_filter: Callable[[str], bool],
                            strip_prefix: Optional[str] = None, verbose: bool = True,
                            rename: Optional[Callable[[str], str]] = None) -> List[Path]:
    """Extract matching members; STORED members are streamed in one Range request each."""
    dest = Path(dest)
    out: List[Path] = []
    sess = _session()
    reader = HttpRangeReader(url, chunk=16 << 20, session=sess)
    with zipfile.ZipFile(reader) as zf:
        members = [m for m in zf.infolist() if not m.is_dir() and member_filter(m.filename)]
        total = sum(m.file_size for m in members)
        if verbose:
            print(f"[zip] {url.rsplit('/', 1)[-1]}: {len(members)} members, {total / 2**30:.2f} GiB")
        for k, m in enumerate(members):
            name = _target_name(m.filename, strip_prefix, rename)
            dst = dest / name
            if dst.exists() and dst.stat().st_size == m.file_size:
                out.append(dst)
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            big = m.file_size > (64 << 20)
            if big and m.compress_type in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED):
                hdr = reader.fetch(m.header_offset, m.header_offset + 29)
                n_name, n_extra = struct.unpack("<HH", hdr[26:30])
                data_start = m.header_offset + 30 + n_name + n_extra
                if m.compress_type == zipfile.ZIP_STORED:
                    _stream_range_to_file(sess, url, data_start, m.file_size, dst)
                else:
                    _stream_deflate_to_file(sess, url, data_start, m.compress_size, m.file_size, dst)
            else:
                with zf.open(m) as src, open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
            out.append(dst)
            if verbose and (m.file_size > (256 << 20) or (k + 1) % 50 == 0 or k + 1 == len(members)):
                print(f"[zip]   {k + 1}/{len(members)} {name} ({m.file_size / 2**20:.0f} MiB)")
    return out


def extract_zip_any(url: str, dest: Path, member_filter: Callable[[str], bool], strip_prefix: Optional[str] = None,
                    tmp_dir: Path = Path("/tmp/k3d_dl"), rename: Optional[Callable[[str], str]] = None) -> List[Path]:
    """Range-based extraction, or full download + local extraction if the server does not support Range."""
    try:
        return extract_from_remote_zip(url, dest, member_filter, strip_prefix, rename=rename)
    except IOError as exc:
        print(f"[zip] range extraction failed ({exc}); downloading the whole archive")
        local = download(url, Path(tmp_dir) / url.rsplit("/", 1)[-1])
        out = []
        with zipfile.ZipFile(local) as zf:
            for m in zf.infolist():
                if m.is_dir() or not member_filter(m.filename):
                    continue
                dst = Path(dest) / _target_name(m.filename, strip_prefix, rename)
                dst.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(m) as src, open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
                out.append(dst)
        local.unlink(missing_ok=True)
        return out


def stream_tar_members(url: str, dest: Path, member_filter: Callable[[str], bool],
                       stop_after_dirs: Optional[int] = None, verbose: bool = True) -> List[Path]:
    """Stream a remote tar, keep matching members. stop_after_dirs: stop once that many sample ids were seen."""
    dest = Path(dest)
    out, seen = [], []
    with _session().get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
            for m in tf:
                parts = Path(m.name).parts
                sid = parts[1] if len(parts) > 2 else (parts[0] if parts else "")
                if sid and (not seen or seen[-1] != sid):
                    if stop_after_dirs is not None and len(seen) >= stop_after_dirs:
                        break
                    seen.append(sid)
                if not m.isfile() or not member_filter(m.name):
                    continue
                dst = dest / m.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                src = tf.extractfile(m)
                with open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
                out.append(dst)
                if verbose and len(out) % 200 == 0:
                    print(f"[tar] {len(out)} files, {len(seen)} samples")
    if verbose:
        print(f"[tar] done: {len(out)} files from {len(seen)} samples")
    return out


In [ ]:
%%writefile /kaggle/working/k3d/calib/common.py
"""Shared helpers: environment, teacher sampling, VAE decode, render-LPIPS exactly as scripts/hackathon/score.py."""

from __future__ import annotations

import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch

TEACHER_DEFAULT = dict(guidance_scale=7.0, cfg_range=(0.5, 1.0), rescale_t=3.0, num_inference_steps=50)


def setup_env(repo_root: str, compat_root: str, verbose: bool = True, need_mesh: bool = True) -> Dict[str, str]:
    os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")  # avoid recompiles of @torch.compile helpers per shape
    try:
        import torch._dynamo
        torch._dynamo.config.disable = True
    except Exception:  # noqa: BLE001
        pass
    if compat_root not in sys.path:
        sys.path.insert(0, compat_root)
    import t4compat
    rep = t4compat.install(repo_root, verbose=verbose)
    rep.update(t4compat.prepare_ops(verbose=verbose, need_mesh=need_mesh))
    hack = str(Path(repo_root) / "scripts" / "hackathon")
    if hack not in sys.path:
        sys.path.append(hack)
    return rep


def load_models(weights_root: str, device, load_vae: bool = True):
    import t4compat
    from kandi3d_image23d.models.modeling_lat_vae import LatVaeModel
    dtype = torch.float32 if str(device).startswith("cpu") else torch.float16
    teacher = t4compat.load_lat_dit_fp16(str(Path(weights_root) / "lat_dit"), device, dtype=dtype)
    vae = None
    if load_vae:
        vae = LatVaeModel.from_pretrained(str(Path(weights_root) / "lat_vae"))
        vae.to(torch.device(device))
        vae.core.device = torch.device(device)  # used by decode() for the invalid-mesh branch
        vae.eval()
    return teacher, vae


def load_sample(sample_dir: Path, device, view: str = "4view_000"):
    cache = sample_dir / "cache" if (sample_dir / "cache").is_dir() else sample_dir
    vox = np.load(cache / "vox_idxi.npz")["vox_idxi"]
    d = np.load(cache / f"{view}_dinov3.npz")
    vox_t = torch.tensor(vox, dtype=torch.int32, device=device)
    feats = torch.tensor(d["feats"], device=device)
    token = torch.tensor(d["token"], device=device)
    return vox_t, feats, token


def nfe_of(steps: int, guidance: float, cfg_range=(0.5, 1.0), rescale_t: float = 3.0) -> int:
    t = torch.linspace(1, 0, steps + 1, dtype=torch.float32)
    t = rescale_t * t / (1 + (rescale_t - 1) * t)
    n_cfg = sum(1 for x in t[:-1].tolist() if guidance > 0 and cfg_range[0] <= x <= cfg_range[1])
    return steps + n_cfg


@torch.no_grad()
def run_teacher(model, vox, feats, token, *, steps: int, guidance: float = 7.0, cfg_range=(0.5, 1.0),
                rescale_t: float = 3.0, seed: int = 42, device="cuda") -> Tuple[torch.Tensor, float]:
    """Same call path as LatDitStage.run -> generate(): returns (latent [N,16] fp32, seconds)."""
    from kandi3d_image23d.modeling_utils import stage_autocast
    from kandi3d_image23d.runtime import FlowMatchSamplingParams, LatDitConditioning
    dev = torch.device(device)
    cond = LatDitConditioning(dino_features=feats, dino_token=token, batch_size=1)
    params = FlowMatchSamplingParams(guidance_scale=guidance, cfg_range=tuple(cfg_range), rescale_t=rescale_t,
                                     num_inference_steps=steps)
    gen = torch.Generator(dev).manual_seed(seed)
    with stage_autocast("lat_dit", dev):
        out = model.generate([vox], conditioning=cond, sampling=params, generator=gen)
    secs = float(model.last_run_metadata["elapsed_seconds"])
    lat = out[0].float()
    if not torch.isfinite(lat).all():
        raise FloatingPointError("non-finite latent (fp16 overflow?)")
    return lat, secs


@torch.no_grad()
def decode_mesh(vae, latent: torch.Tensor, vox: torch.Tensor, device="cuda") -> Tuple[np.ndarray, np.ndarray]:
    """Identical to LatDitStage.run decode branch (postprocess_mesh(n_faces=600000) + rotate_vertices)."""
    from base_models.mesh_utils import postprocess_mesh
    from kandi3d_image23d.lat_dit_stage import rotate_vertices
    from kandi3d_image23d.modeling_utils import stage_autocast
    dev = torch.device(device)
    with stage_autocast("lat_vae", dev):
        mesh_out = vae([latent], [vox])
    verts, faces = postprocess_mesh(mesh_out.vertices, mesh_out.triangles, n_faces=600000)
    verts, faces = verts.float(), faces.int()
    verts = rotate_vertices(verts)
    return verts.cpu().numpy().astype(np.float32), faces.cpu().numpy().astype(np.int32)


def save_mesh(path: Path, verts: np.ndarray, faces: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, vertices=verts, faces=faces)


_LPIPS = None


def mesh_lpips(path_ref: Path, path_pred: Path) -> float:
    """scripts/hackathon/score.py::compare_two_meshes with the same defaults (res 512, 4 views, squeeze)."""
    global _LPIPS
    from score import compare_two_meshes
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    if _LPIPS is None:
        _LPIPS = LearnedPerceptualImagePatchSimilarity(net_type="squeeze")
    return float(compare_two_meshes(str(path_ref), str(path_pred), res=512, num_views=4, lpips=_LPIPS))


def latent_rel_mse(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(((a - b) ** 2).mean() / (b ** 2).mean().clamp(min=1e-12))


def quality_code(lp: float, q_min: float = 0.25) -> float:
    return 0.0 if (lp is None or not np.isfinite(lp) or lp > q_min) else 1.0 - lp


@torch.no_grad()
def activation_report(model, vox, feats, token, steps: int = 1, device="cuda") -> Dict[str, float]:
    """Max |activation| of fp16 tensors inside the teacher (fp16 overflows at 65504)."""
    stats: Dict[str, float] = {}
    hooks = []
    names = ("to_qkv", "ffn0", "ffn1", "cond_ffn0", "cond_ffn1", "to_out_cross", "cond_to_qkv")
    for bi, block in enumerate(model.core.flow.blocks):
        for nm in names:
            mod = getattr(block, nm, None)
            if mod is None:
                continue
            key = f"{nm}"

            def hook(_m, _inp, out, key=key):
                t = out.feats if hasattr(out, "feats") else out
                v = float(t.detach().abs().float().max())
                stats[key] = max(stats.get(key, 0.0), v)
            hooks.append(mod.register_forward_hook(hook))
    try:
        lat, _ = run_teacher(model, vox, feats, token, steps=steps, device=device)
        stats["latent_absmax"] = float(lat.abs().max())
    finally:
        for h in hooks:
            h.remove()
    return stats


In [ ]:
%%writefile /kaggle/working/k3d/gen/stream_chunk.py
"""Stream a remote train_part_XXX.tar and keep only the lat_dit inputs of each object.

For every object it writes <dest>/<id>/cache/{vox_idxi.npz, 4view_000_dinov3.npz} and then an empty
<dest>/<id>/.ready marker, so GPU workers can start while the download is still running.
On a network error the stream is resumed with an HTTP Range request from the next tar header.
When the whole tar has been read it writes <dest>/.done (with the number of objects).
"""

from __future__ import annotations

import argparse
import re
import shutil
import sys
import tarfile
import time
from pathlib import Path

import requests

NEEDED = ("vox_idxi.npz", "4view_000_dinov3.npz")
ID_RE = re.compile(r"(?:^|.*/)([0-9a-f]{32})/cache/([^/]+)$")


class _Enough(Exception):
    pass


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True)
    ap.add_argument("--dest", required=True)
    ap.add_argument("--max-objects", type=int, default=None)
    args = ap.parse_args()

    dest = Path(args.dest)
    dest.mkdir(parents=True, exist_ok=True)
    have = {}
    n_ready = sum(1 for p in dest.glob("*/.ready"))
    t0 = time.time()
    base = 0          # absolute byte offset where the current HTTP stream starts
    resume_at = 0     # absolute offset of the next tar header to read
    n_bytes = 0
    for attempt in range(8):
        headers = {"Range": f"bytes={resume_at}-"} if resume_at else {}
        try:
            with requests.get(args.url, stream=True, timeout=600, headers=headers) as r:
                r.raise_for_status()
                if resume_at and r.status_code != 206:
                    resume_at = 0  # server ignored Range: start over (finished objects are skipped)
                base = resume_at
                r.raw.decode_content = True
                with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
                    for m in tf:
                        member_end = base + m.offset_data + ((m.size + 511) // 512) * 512
                        resume_at = base + m.offset  # a failure inside this member restarts at its header
                        n_bytes = member_end
                        mt = ID_RE.match(m.name)
                        if not m.isfile() or mt is None or mt.group(2) not in NEEDED:
                            resume_at = member_end
                            continue
                        sid, fname = mt.group(1), mt.group(2)
                        if (dest / sid / ".ready").exists():
                            resume_at = member_end
                            continue
                        out = dest / sid / "cache" / fname
                        out.parent.mkdir(parents=True, exist_ok=True)
                        tmp = out.with_name(out.name + ".part")
                        with open(tmp, "wb") as f:
                            shutil.copyfileobj(tf.extractfile(m), f, length=8 << 20)
                        tmp.rename(out)
                        resume_at = member_end
                        have.setdefault(sid, set()).add(fname)
                        if all(x in have[sid] or (dest / sid / "cache" / x).exists() for x in NEEDED):
                            (dest / sid / ".ready").touch()
                            n_ready += 1
                            if n_ready % 50 == 0:
                                el = time.time() - t0
                                print(f"[stream] {n_ready} objects ready, {n_bytes / 2**30:.1f} GiB read, "
                                      f"{n_bytes / 2**20 / max(el, 1):.0f} MiB/s", flush=True)
                            if args.max_objects and n_ready >= args.max_objects:
                                raise _Enough
            break
        except _Enough:
            break
        except Exception as exc:  # noqa: BLE001
            print(f"[stream] error {type(exc).__name__}: {exc}; resume at {resume_at / 2**30:.2f} GiB "
                  f"(retry {attempt + 1}/8)", flush=True)
            time.sleep(20)
    n_total = sum(1 for p in dest.glob("*/.ready"))
    (dest / ".done").write_text(str(n_total))
    print(f"[stream] done: {n_total} objects in {(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/k3d/gen/run_gen.py
"""GPU worker: teacher targets for the student, for every object that the streamer marked ready.

Each worker takes the objects with crc32(id) % nshards == shard, so two GPUs never collide.
Phase 1 (cheap, 2 NFE, every object up to --max-vox1 voxels, as soon as it is downloaded):
  <out>/x0/<id>.npz   x0c, x0u, z  ([N,16] fp16, rows in vox_idxi order)
    x0c  one-step prediction at t=1 from the conditional pass:   denorm(z - v_cond(z, t=1))
    x0u  one-step prediction at t=1 from the zero-context pass:  denorm(z - v_uncond(z, t=1))
    z    the initial noise (normalized space; T4 CUDA generator, seed 42)
  The 1-step teacher with guidance w equals x0c + w * (x0c - x0u) (the repo applies pred + w * (pred - neg_pred)).
Phase 2 (after phase 1 is complete, objects up to --max-vox voxels, pseudo-random order, until the budget ends):
  <out>/lat8/<id>.npz latent [N,16] fp16 = what generate() returns with --steps steps (denormalized)
Plus rows in <out>/index_shard{k}.csv. Existing outputs are skipped (resume between sessions).
"""

from __future__ import annotations

import os

os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

import argparse
import csv
import sys
import time
import zlib
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent / "calib"))


class FirstStepCapture:
    """Forward hook on core.flow that records the calls made at t=1 (timestep input 1000): the conditional pass
    and, when CFG is active at t=1, the zero-context pass. At t=1 the sampler state x_t is the pure noise z."""

    def __init__(self, core):
        self.core = core
        self.calls = []
        self.armed = False
        self.handle = core.flow.register_forward_hook(self._hook)

    def _hook(self, module, args, output):
        if not self.armed:
            return
        t = float(args[1].reshape(-1)[0])
        if t < 999.5 or len(self.calls) >= 2:
            self.armed = False  # past the first step: stop checking (avoids a sync per call)
            return
        self.calls.append((args[0].feats.detach().float().clone(), output.feats.detach().float().clone()))

    def arm(self) -> None:
        self.calls, self.armed = [], True

    def collect(self) -> dict:
        self.armed = False
        core = self.core
        std = core.std.reshape(1, -1).float()
        mean = core.mean.reshape(1, -1).float()
        scale = float(core.latent_scale)

        def den(x):
            return (x * std + mean) * scale

        out = {}
        if self.calls:
            z, v_c = self.calls[0]
            out["z"], out["x0c"] = z, den(z - v_c)
        if len(self.calls) > 1:
            z2, v_u = self.calls[1]
            out["x0u"] = den(z2 - v_u)
        self.calls = []
        return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--compat", required=True)
    ap.add_argument("--weights", required=True, help="dir with lat_dit/")
    ap.add_argument("--watch", required=True, help="dir with <id>/cache/* and <id>/.ready")
    ap.add_argument("--out", required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--steps", type=int, default=8, help="phase 2 sampler steps")
    ap.add_argument("--guidance", type=float, default=7.0)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--max-vox", type=int, default=20000, help="phase 2 size limit")
    ap.add_argument("--max-vox1", type=int, default=45000, help="phase 1 size limit")
    ap.add_argument("--no-phase2", action="store_true")
    ap.add_argument("--max-hours", type=float, default=9.0)
    ap.add_argument("--device", default="cuda")
    args = ap.parse_args()

    import numpy as np
    import torch
    from common import setup_env, load_models, load_sample, run_teacher, nfe_of

    t_start = time.time()
    deadline = t_start + args.max_hours * 3600
    out = Path(args.out)
    d1, d2 = out / "x0", out / "lat8"
    d1.mkdir(parents=True, exist_ok=True)
    d2.mkdir(parents=True, exist_ok=True)
    watch = Path(args.watch)
    log = open(out / f"log_shard{args.shard}.txt", "a", buffering=1)

    def say(msg: str) -> None:
        print(msg, flush=True)
        log.write(msg + "\n")

    rep = setup_env(args.repo, args.compat, verbose=False, need_mesh=False)
    dev = args.device
    teacher, _ = load_models(args.weights, dev, load_vae=False)
    cap = FirstStepCapture(teacher.core)
    gpu = torch.cuda.get_device_name(0) if dev.startswith("cuda") else dev
    say(f"[gen {args.shard}] env {rep} | gpu {gpu} | phase1: 1 step (nfe {nfe_of(1, args.guidance)}) N<={args.max_vox1} | "
        f"phase2: {args.steps} steps (nfe {nfe_of(args.steps, args.guidance)}) N<={args.max_vox} | budget {args.max_hours} h")

    idx_path = out / f"index_shard{args.shard}.csv"
    new_file = not idx_path.exists()
    fidx = open(idx_path, "a", newline="", buffering=1)
    w = csv.writer(fidx)
    if new_file:
        w.writerow(["id", "n_vox", "phase", "gen_s", "status"])

    def mine(sid: str) -> bool:
        return zlib.crc32(sid.encode()) % args.nshards == args.shard

    def save(path: Path, **arrays) -> None:
        tmp = path.with_name(path.stem + ".tmp.npz")
        np.savez_compressed(tmp, **arrays)
        tmp.rename(path)

    done1 = {p.stem for p in d1.glob("*.npz") if not p.stem.endswith(".tmp")}
    done2 = {p.stem for p in d2.glob("*.npz") if not p.stem.endswith(".tmp")}
    skip1, skip2, n_of = set(), set(), {}
    stats = {1: [0, 0, 0.0], 2: [0, 0, 0.0]}  # new objects, voxels, seconds

    def report(phase: int) -> None:
        k, vox, sec = stats[phase]
        if k % 25 == 0:
            el = (time.time() - t_start) / 3600
            say(f"[gen {args.shard}] phase{phase}: {k} new | x0 {len(done1)} lat8 {len(done2)} | "
                f"{sec / max(vox, 1) * 1000:.2f} s per 1k vox | {el:.2f} h")

    def fail(sid: str, phase: int, exc: Exception) -> None:
        (skip1 if phase == 1 else skip2).add(sid)
        w.writerow([sid, n_of.get(sid, ""), phase, "", f"error: {type(exc).__name__}: {exc}"[:200]])
        say(f"[gen {args.shard}] phase{phase} {sid} failed: {type(exc).__name__}: {exc}")
        if dev.startswith("cuda"):
            torch.cuda.empty_cache()

    # ---- phase 1: one-step predictions (2 NFE) for every object as soon as it is downloaded
    while time.time() < deadline:
        ready = [p.parent.name for p in watch.glob("*/.ready")]
        todo = sorted((s for s in ready if mine(s) and s not in done1 and s not in skip1),
                      key=lambda s: (watch / s / ".ready").stat().st_mtime)
        if not todo:
            if (watch / ".done").exists():
                break
            time.sleep(15)
            continue
        for sid in todo:
            if time.time() >= deadline:
                break
            try:
                vox, feats, token = load_sample(watch / sid, dev)
                n = n_of[sid] = int(vox.shape[0])
                if n > args.max_vox1 or n == 0:
                    skip1.add(sid)
                    w.writerow([sid, n, 1, "", "skipped_size"])
                    continue
                cap.arm()
                _, secs = run_teacher(teacher, vox, feats, token, steps=1, guidance=args.guidance, seed=args.seed, device=dev)
                c = {k: v.half().cpu().numpy() for k, v in cap.collect().items()}
                if set(c) != {"z", "x0c", "x0u"} or any(v.shape != (n, 16) for v in c.values()):
                    raise RuntimeError(f"capture failed: {[(k, v.shape) for k, v in c.items()]}")
                save(d1 / f"{sid}.npz", **c, n_vox=n, guidance=args.guidance, seed=args.seed)
                done1.add(sid)
                w.writerow([sid, n, 1, f"{secs:.2f}", "ok"])
                st = stats[1]; st[0] += 1; st[1] += n; st[2] += secs
                report(1)
            except Exception as exc:  # noqa: BLE001
                fail(sid, 1, exc)
    say(f"[gen {args.shard}] phase1 finished: {stats[1][0]} new, {len(done1)} total, {(time.time() - t_start) / 3600:.2f} h")

    # ---- phase 2: full sampler (default 8 steps) for objects <= max_vox, pseudo-random order
    if not args.no_phase2:
        cand = [s for s in done1 if mine(s) and s not in done2]
        for s in cand:
            if s not in n_of:
                with np.load(d1 / f"{s}.npz") as d:
                    n_of[s] = int(d["n_vox"])
        todo = sorted((s for s in cand if 0 < n_of[s] <= args.max_vox), key=lambda s: zlib.crc32((s + "p2").encode()))
        for sid in todo:
            if time.time() >= deadline:
                break
            try:
                vox, feats, token = load_sample(watch / sid, dev)
                lat, secs = run_teacher(teacher, vox, feats, token, steps=args.steps, guidance=args.guidance,
                                        seed=args.seed, device=dev)
                save(d2 / f"{sid}.npz", latent=lat.half().cpu().numpy(), n_vox=n_of[sid], steps=args.steps,
                     guidance=args.guidance, seed=args.seed)
                done2.add(sid)
                w.writerow([sid, n_of[sid], 2, f"{secs:.2f}", "ok"])
                st = stats[2]; st[0] += 1; st[1] += n_of[sid]; st[2] += secs
                report(2)
            except Exception as exc:  # noqa: BLE001
                fail(sid, 2, exc)
    reason = "budget" if time.time() >= deadline else "no more objects"
    say(f"[gen {args.shard}] finished ({reason}): x0 {len(done1)} | lat8 {len(done2)} | "
        f"{(time.time() - t_start) / 3600:.2f} h")


if __name__ == "__main__":
    main()


In [ ]:
# 3. Код baseline и веса lat_dit (~5–10 мин)
sys.path.insert(0, str(K3D / "calib"))
import importlib, k3d_data as D
importlib.reload(D)
names = [m.filename for m in D.list_remote_zip(D.URLS["code"])]
marker = "kandi3d_image23d/__init__.py"
root = next(n[: -len(marker)] for n in names if n.endswith(marker))
D.extract_zip_any(D.URLS["code"], DATA,
                  lambda n: n.startswith(root) and "__MACOSX" not in n and "/.git/" not in n and not n.endswith("/"),
                  rename=lambda n: "3dgen-main/" + n[len(root):])
D.extract_zip_any(D.URLS["weights"], WEIGHTS,
                  lambda n: "lat_dit/" in n and "__MACOSX" not in n and not n.endswith("/"),
                  rename=lambda n: n[n.index("lat_"):])
print("code:", (REPO / "kandi3d_image23d").exists(), "| weights:",
      {str(p.relative_to(WEIGHTS)): f"{p.stat().st_size / 2**20:.0f} MiB" for p in WEIGHTS.rglob("*") if p.is_file()})


In [ ]:
# 4. Компиляция CUDA-опов (curope, libsparse; libmesh не нужен) ~5 мин
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "{}.{}".format(*torch.cuda.get_device_capability()))
os.environ.setdefault("MAX_JOBS", "4")
os.environ["TORCH_EXTENSIONS_DIR"] = str(WORK / "torch_ext")
os.environ["TORCHDYNAMO_DISABLE"] = "1"
sys.path.insert(0, str(K3D))
from common import setup_env
t0 = time.time()
ENV = setup_env(str(REPO), str(K3D), need_mesh=False)
print(f"setup {time.time() - t0:.0f}s ->", ENV)
if PREV_TARGETS:
    n_prev = 0
    for sub in ("x0", "lat8"):
        (TARGETS / sub).mkdir(parents=True, exist_ok=True)
        for f in (Path(PREV_TARGETS) / sub).glob("*.npz"):
            if not (TARGETS / sub / f.name).exists():
                shutil.copy(f, TARGETS / sub / f.name); n_prev += 1
    print("продолжение: скопировано готовых файлов:", n_prev)


In [ ]:
# 5. Загрузка потоком + генерация на всех GPU. Прогресс каждые 5 минут.
url = D.URLS["train"].format(CHUNK)
streamer = subprocess.Popen([sys.executable, str(K3D / "gen" / "stream_chunk.py"), "--url", url, "--dest", str(STREAM)],
                            stdout=open(WORK / "stream_log.txt", "w"), stderr=subprocess.STDOUT)
ngpu = torch.cuda.device_count()
workers = []
for g in range(ngpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    cmd = [sys.executable, str(K3D / "gen" / "run_gen.py"), "--repo", str(REPO), "--compat", str(K3D),
           "--weights", str(WEIGHTS), "--watch", str(STREAM), "--out", str(TARGETS), "--shard", str(g),
           "--nshards", str(ngpu), "--steps", str(STEPS), "--max-vox", str(MAX_VOX), "--max-vox1", str(MAX_VOX1),
           "--max-hours", str(MAX_HOURS)] + ([] if PHASE2 else ["--no-phase2"])
    workers.append(subprocess.Popen(cmd, env=env, stdout=open(WORK / f"gen_stdout_{g}.txt", "w"), stderr=subprocess.STDOUT))

def tail(path, n=1):
    try:
        lines = Path(path).read_text().strip().splitlines()
        return lines[-n:] if lines else ["(пусто)"]
    except FileNotFoundError:
        return ["(нет файла)"]

t0 = time.time()
while any(w.poll() is None for w in workers):
    time.sleep(300)
    if streamer.poll() is not None and not (STREAM / ".done").exists():
        (STREAM / ".done").write_text("streamer exited")  # чтобы воркеры не ждали вечно
    n_ready = sum(1 for _ in STREAM.glob("*/.ready"))
    n_x0 = sum(1 for _ in (TARGETS / "x0").glob("*.npz"))
    n_l8 = sum(1 for _ in (TARGETS / "lat8").glob("*.npz"))
    print(f"[{(time.time() - t0) / 60:6.1f} мин] скачано {n_ready}, x0 {n_x0}, lat8 {n_l8} | "
          + " | ".join(tail(TARGETS / f"log_shard{g}.txt")[0][:90] for g in range(ngpu)), flush=True)
if streamer.poll() is None:
    streamer.terminate()  # бюджет исчерпан раньше, чем докачался чанк
print("воркеры завершились:", [w.returncode for w in workers], "| загрузчик:", streamer.poll())
print("\n".join(tail(WORK / "stream_log.txt", 3)))
for g in range(ngpu):
    print(f"--- gpu{g} ---"); print("\n".join(tail(WORK / f"gen_stdout_{g}.txt", 8)))


In [ ]:
# 6. Итог + архив для скачивания
import pandas as pd, numpy as np
for sub in ("x0", "lat8"):
    ex = next((TARGETS / sub).glob("*.npz"), None)
    if ex is not None:
        with np.load(ex) as d:
            print(f"пример {sub}/{ex.name}:", {k: d[k].shape for k in d.files})
rows = [pd.read_csv(f) for f in TARGETS.glob("index_shard*.csv")]
idx = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["id", "n_vox", "phase", "gen_s", "status"])
idx.to_csv(TARGETS / "index.csv", index=False)
for ph in (1, 2):
    g = idx[idx.phase == ph]
    ok = g[g.status == "ok"].astype({"gen_s": float, "n_vox": float})
    print(f"фаза {ph}: файлов {sum(1 for _ in (TARGETS / ('x0' if ph == 1 else 'lat8')).glob('*.npz'))} | в этом запуске ok {len(ok)}"
          f" | пропущено по размеру {int((g.status == 'skipped_size').sum())}"
          f" | ошибок {int(g.status.astype(str).str.startswith('error').sum())}"
          + (f" | {ok.gen_s.sum() / ok.n_vox.sum() * 1000:.2f} с на 1k вокселей, GPU {ok.gen_s.sum() / 3600:.1f} ч" if len(ok) else ""))
shutil.make_archive(str(WORK / "targets"), "zip", root_dir=WORK, base_dir="targets")
print("архив:", WORK / "targets.zip", f"{(WORK / 'targets.zip').stat().st_size / 2**20:.0f} MiB")


## Дальше
Скачай `targets.zip` и положи в папку проекта. Цели + входы train-чанка 001 с Мака = обучающая выборка для крошечного студента.
Если фаза 2 обработала не все объекты — запусти ещё раз с `PREV_TARGETS` (см. верх ноутбука).